# Part 8 — Answer Generation & Query Graph

This notebook explores the generation layer of the GraphRAG pipeline:

| Module | Responsibility |
|---|---|
| `answer_generator.py` | `format_context`, `generate_answer`, `web_search_fallback` |
| `hallucination_grader.py` | `grade_answer` → `GraderDecision` |
| `query_graph.py` | LangGraph `StateGraph` orchestrator |

**Architecture:**
```
user_query
    │
    ▼
hybrid_retrieval ──► reranking ──► answer_generation ──► hallucination_grader
                                          ▲                        │
                                   (regenerate)            (pass) ─► finalise ──► END
                                          │            (web_search) ─► web_search ──► END
                                          └────────────────────────┘
```

All cells below run **offline** — real LLMs and Neo4j are replaced with `MagicMock`.

In [1]:
import sys
from pathlib import Path

# Add project root to sys.path
repo_root = Path.cwd().parents[1]
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"Project root: {repo_root}")

Project root: /home/marcantoniolopez/Documenti/github/thesis


## §1 — `format_context(chunks)`

`format_context()` converts a list of `RetrievedChunk` objects into the numbered
string that is injected into the LLM prompt. Each entry shows the text, node type,
and similarity score — giving the model provenance information for every fact.

In [2]:
from src.generation.answer_generator import format_context
from src.models.schemas import RetrievedChunk

chunks = [
    RetrievedChunk(
        node_id="n1",
        text="Customer: A person who purchases products from the company.",
        node_type="BusinessConcept",
        score=0.97,
        source_type="vector",
    ),
    RetrievedChunk(
        node_id="n2",
        text="Order: A purchase transaction. Contains one or more products.",
        node_type="BusinessConcept",
        score=0.88,
        source_type="bm25",
    ),
    RetrievedChunk(
        node_id="n3",
        text="TB_ORD: Database table storing orders. Columns: id, customer_id, total.",
        node_type="PhysicalTable",
        score=0.71,
        source_type="graph",
    ),
]

context = format_context(chunks)
print(context)


[1] Customer: A person who purchases products from the company.  [type=BusinessConcept, score=0.970]
[2] Order: A purchase transaction. Contains one or more products.  [type=BusinessConcept, score=0.880]
[3] TB_ORD: Database table storing orders. Columns: id, customer_id, total.  [type=PhysicalTable, score=0.710]


In [3]:
# Empty list → placeholder so the LLM still gets a valid prompt
print(format_context([]))

(no context retrieved)


## §2 — `generate_answer(query, chunks, llm, critique=None)`

The answer generator uses `ANSWER_SYSTEM` / `ANSWER_USER` at temperature 0.3.
When a `critique` is supplied on retry, `ANSWER_WITH_CRITIQUE_USER` is used instead,
injecting the grader's feedback so the model corrects specific unsupported claims.

In [4]:
from unittest.mock import MagicMock

from src.generation.answer_generator import generate_answer

# Mock LLM — returns a fixed string via the .content attribute
mock_llm = MagicMock()
mock_llm.invoke.return_value.content = (
    "A Customer is a person who purchases products from the company. "
    "Orders are stored in the TB_ORD table with columns: id, customer_id, total."
)

answer = generate_answer(
    query="What is a Customer and where are orders stored?",
    chunks=chunks,
    llm=mock_llm,
)
print("Answer (no critique):")
print(answer)

{"ts": "2026-03-12T17:08:05", "logger": "src.generation.answer_generator", "level": "INFO", "message": "Answer generated (139 chars)."}


Answer (no critique):
A Customer is a person who purchases products from the company. Orders are stored in the TB_ORD table with columns: id, customer_id, total.


In [5]:
# Retry with a critique injected by the Hallucination Grader
critique = (
    "The column 'total' is not mentioned in any retrieved context chunk. "
    "Reformulate the answer omitting that column."
)

mock_llm.invoke.return_value.content = (
    "A Customer is a person who purchases products. "
    "Orders are stored in TB_ORD with columns: id and customer_id."
)

answer_v2 = generate_answer(
    query="What is a Customer and where are orders stored?",
    chunks=chunks,
    llm=mock_llm,
    critique=critique,
)
print("Answer (with critique):")
print(answer_v2)

# Verify that the ANSWER_WITH_CRITIQUE_USER template was used
call_args = mock_llm.invoke.call_args[0][0]
human_content = call_args[-1].content
print(f"\nCritique injected in prompt: {'Reformulate' in human_content}")

{"ts": "2026-03-12T17:08:05", "logger": "src.generation.answer_generator", "level": "INFO", "message": "Answer generated (108 chars)."}


Answer (with critique):
A Customer is a person who purchases products. Orders are stored in TB_ORD with columns: id and customer_id.

Critique injected in prompt: True


## §3 — `web_search_fallback(query)`

When the grader sets `action="web_search"`, Tavily is tried first. On `ImportError`
or `RuntimeError` (no API key), DuckDuckGo is used. If both fail, a safe fallback
message is returned. All results are prefixed with `[Source: Web Search]`.

In [6]:
from unittest.mock import MagicMock, patch

from src.generation.answer_generator import web_search_fallback

_TAVILY = "langchain_community.tools.tavily_search.TavilySearchResults"
_DDG = "langchain_community.tools.DuckDuckGoSearchRun"

# Scenario 1: Tavily succeeds
mock_tavily_instance = MagicMock()
mock_tavily_instance.run.return_value = "Tavily result: Customer data management ..."
mock_tavily_cls = MagicMock(return_value=mock_tavily_instance)

with patch(_TAVILY, mock_tavily_cls):
    result = web_search_fallback("What is a Customer?")
print("Tavily success:")
print(result)

{"ts": "2026-03-12T17:08:05", "logger": "src.generation.answer_generator", "level": "INFO", "message": "Web search fallback triggered for query: 'What is a Customer?'."}


Tavily success:
[Source: Web Search] 


In [7]:
import builtins
real_import = builtins.__import__

def import_blocker(name, *args, **kwargs):
    if name == "langchain_community":
        raise ImportError("langchain_community not available")
    return real_import(name, *args, **kwargs)

# Scenario 2: Tavily unavailable → DuckDuckGo fallback
mock_ddg_instance = MagicMock()
mock_ddg_instance.run.return_value = "DDG result: Customer definition ..."
mock_ddg_cls = MagicMock(return_value=mock_ddg_instance)

with patch("builtins.__import__", side_effect=import_blocker):
    with patch(_DDG, mock_ddg_cls):
        try:
            result2 = web_search_fallback("What is a Customer?")
        except Exception:
            result2 = "[Source: Web Search] Unable to retrieve external information."
print("DDG fallback result:")
print(result2)

{"ts": "2026-03-12T17:08:05", "logger": "src.generation.answer_generator", "level": "INFO", "message": "Web search fallback triggered for query: 'What is a Customer?'."}
/home/marcantoniolopez/Documenti/github/thesis/src/generation/answer_generator.py:120: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily = TavilySearchResults(max_results=3)


DDG fallback result:
[Source: Web Search] 


In [8]:
# Scenario 3: Both fail → graceful degradation
mock_tavily_fail = MagicMock()
mock_tavily_fail.run.side_effect = RuntimeError("No TAVILY_API_KEY")
mock_tavily_fail_cls = MagicMock(return_value=mock_tavily_fail)

mock_ddg_fail = MagicMock()
mock_ddg_fail.run.side_effect = RuntimeError("DDG rate limited")
mock_ddg_fail_cls = MagicMock(return_value=mock_ddg_fail)

with patch(_TAVILY, mock_tavily_fail_cls), patch(_DDG, mock_ddg_fail_cls):
    result3 = web_search_fallback("What is a Customer?")
print("Both fail → graceful message:")
print(result3)

{"ts": "2026-03-12T17:08:06", "logger": "src.generation.answer_generator", "level": "INFO", "message": "Web search fallback triggered for query: 'What is a Customer?'."}


Both fail → graceful message:
[Source: Web Search] 


## §4 — `grade_answer(query, answer, chunks, llm) → GraderDecision`

The hallucination grader applies `GRADER_SYSTEM` / `GRADER_USER` at temperature 0.0.
It returns a `GraderDecision` Pydantic model with three verdicts:

| `action` | `grounded` | Meaning |
|---|---|---|
| `"pass"` | `True` | All claims supported — accept answer |
| `"regenerate"` | `False` | Specific unsupported claims — retry with critique |
| `"web_search"` | `False` | Context irrelevant — fall back to web search |

In [9]:
import json
from unittest.mock import MagicMock

from src.generation.hallucination_grader import grade_answer
from src.models.schemas import GraderDecision

def mock_grader_llm(verdict: dict) -> MagicMock:
    llm = MagicMock()
    llm.invoke.return_value.content = json.dumps(verdict)
    return llm

query = "What is a Customer?"
answer = "A Customer is a person who purchases products."

# Verdict 1: pass — answer is fully grounded
grader_pass = mock_grader_llm({"grounded": True, "critique": None, "action": "pass"})
decision_pass = grade_answer(query, answer, chunks, grader_pass)
print(f"Verdict 'pass': grounded={decision_pass.grounded}, action={decision_pass.action}")

{"ts": "2026-03-12T17:08:06", "logger": "src.generation.hallucination_grader", "level": "INFO", "message": "Grader verdict: grounded=True, action=pass, critique=''."}


Verdict 'pass': grounded=True, action=pass


In [10]:
# Verdict 2: regenerate — specific unsupported claim found
grader_regen = mock_grader_llm({
    "grounded": False,
    "critique": "The claim about 'premium membership' is not in any retrieved chunk.",
    "action": "regenerate",
})
decision_regen = grade_answer(query, answer, chunks, grader_regen)
print(f"Verdict 'regenerate': grounded={decision_regen.grounded}")
print(f"  critique: {decision_regen.critique}")
print(f"  action:   {decision_regen.action}")

{"ts": "2026-03-12T17:08:06", "logger": "src.generation.hallucination_grader", "level": "INFO", "message": "Grader verdict: grounded=False, action=regenerate, critique=\"The claim about 'premium membership' is not in any retrieved chunk.\"."}


Verdict 'regenerate': grounded=False
  critique: The claim about 'premium membership' is not in any retrieved chunk.
  action:   regenerate


In [11]:
# Verdict 3: web_search — context entirely irrelevant
grader_web = mock_grader_llm({"grounded": False, "critique": None, "action": "web_search"})
decision_web = grade_answer(query, answer, chunks, grader_web)
print(f"Verdict 'web_search': action={decision_web.action}")

{"ts": "2026-03-12T17:08:07", "logger": "src.generation.hallucination_grader", "level": "INFO", "message": "Grader verdict: grounded=False, action=web_search, critique=''."}


Verdict 'web_search': action=web_search


In [12]:
# Graceful degradation: LLM failure → conservative 'pass'
grader_fail = MagicMock()
grader_fail.invoke.side_effect = RuntimeError("LLM timeout")
decision_degraded = grade_answer(query, answer, chunks, grader_fail)
print(f"LLM failure → fallback: grounded={decision_degraded.grounded}, action={decision_degraded.action}")

{"ts": "2026-03-12T17:08:07", "logger": "src.generation.hallucination_grader", "level": "WARNING", "message": "Grader LLM call failed (LLM timeout) \u2014 defaulting to pass."}


LLM failure → fallback: grounded=True, action=pass


In [13]:
# Inconsistency correction: grounded=True but action='regenerate' → auto-corrected to 'pass'
grader_inconsistent = mock_grader_llm({"grounded": True, "critique": "some critique", "action": "regenerate"})
decision_corrected = grade_answer(query, answer, chunks, grader_inconsistent)
print(f"Inconsistency corrected: grounded={decision_corrected.grounded}, action={decision_corrected.action}")

{"ts": "2026-03-12T17:08:07", "logger": "src.generation.hallucination_grader", "level": "WARNING", "message": "Grader inconsistency (grounded=True but action=regenerate) \u2014 forcing pass."}


Inconsistency corrected: grounded=True, action=pass


## §5 — `_route_after_grader(state)` — Conditional Routing

This is the conditional edge function used by LangGraph after the grader node.
It reads `state["grader_decision"].action` and returns the name of the next node.

In [14]:
from src.generation.query_graph import _route_after_grader
from src.models.schemas import GraderDecision

cases = [
    ("pass",        GraderDecision(grounded=True,  critique=None,        action="pass")),
    ("regenerate",  GraderDecision(grounded=False, critique="Too vague.", action="regenerate")),
    ("web_search",  GraderDecision(grounded=False, critique=None,        action="web_search")),
]

print(f"{'Input action':<14} → {'Next node'}")
print("-" * 30)
for label, decision in cases:
    state = {"grader_decision": decision}
    next_node = _route_after_grader(state)
    print(f"{label:<14} → {next_node}")

# Safe defaults
print(f"{'None (missing)':<14} → {_route_after_grader({})}")
print(f"{'unknown':<14} → {_route_after_grader({'grader_decision': GraderDecision(grounded=True, critique=None, action='pass')})}")

/home/marcantoniolopez/Documenti/github/thesis/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/marcantoniolopez/Documenti/github/thesis/.venv/lib/python3.12/site-packages/pydantic_settings/sources/providers/secrets.py:67: UserWarning: directory "/run/secrets" does not exist
  warnings.warn(f'directory "{path}" does not exist')


Input action   → Next node
------------------------------
pass           → finalise
regenerate     → answer_generation
web_search     → web_search
None (missing) → finalise
unknown        → finalise


## §6 — `build_query_graph()` — Graph Compilation

`build_query_graph()` returns a compiled `CompiledStateGraph` with `MemorySaver`
checkpointing. The graph has 6 nodes connected with deterministic and conditional edges.

In [15]:
from unittest.mock import MagicMock, patch

_NEO4J = "src.generation.query_graph.Neo4jClient"
_EMBEDDINGS = "src.generation.query_graph.get_embeddings"
_RERANKER = "src.generation.query_graph.rerank"
_BUILD_NODE_INDEX = "src.generation.query_graph.build_node_index"
_VECTOR = "src.generation.query_graph.vector_search"
_BM25 = "src.generation.query_graph.bm25_search"
_GRAPH = "src.generation.query_graph.graph_traversal"
_MERGE = "src.generation.query_graph.merge_results"
_GEN = "src.generation.query_graph.generate_answer"
_GRADE = "src.generation.query_graph.grade_answer"
_WEB = "src.generation.query_graph.web_search_fallback"
_LLM = "src.generation.query_graph.get_reasoning_llm"

with (
    patch(_NEO4J) as mock_neo4j,
    patch(_EMBEDDINGS, return_value=MagicMock()),
    patch(_BUILD_NODE_INDEX, return_value=[]),
    patch(_VECTOR, return_value=[]),
    patch(_BM25, return_value=[]),
    patch(_GRAPH, return_value=[]),
    patch(_MERGE, return_value=[]),
    patch(_RERANKER, return_value=[]),
    patch(_GEN, return_value="mocked answer"),
    patch(_GRADE, return_value=GraderDecision(grounded=True, critique=None, action="pass")),
    patch(_WEB, return_value="[Source: Web Search] mocked"),
    patch(_LLM, return_value=MagicMock()),
):
    mock_neo4j.return_value.__enter__ = lambda s: MagicMock()
    mock_neo4j.return_value.__exit__ = MagicMock(return_value=False)

    from src.generation.query_graph import build_query_graph
    graph = build_query_graph()

    print(f"Graph type : {type(graph).__name__}")
    print(f"Graph nodes: {list(graph.get_graph().nodes.keys())}")

Graph type : CompiledStateGraph
Graph nodes: ['__start__', 'hybrid_retrieval', 'reranking', 'answer_generation', 'hallucination_grader', 'web_search', 'finalise', '__end__']


## Summary

| Component | Function | Key Design |
|---|---|---|
| `format_context` | Converts `RetrievedChunk` list → numbered string | Score + node_type per entry |
| `generate_answer` | LLM answer at T=0.3 | `ANSWER_USER` or `ANSWER_WITH_CRITIQUE_USER` |
| `web_search_fallback` | Tavily → DuckDuckGo → graceful degradation | Lazy imports; `[Source: Web Search]` prefix |
| `grade_answer` | Self-RAG audit at T=0.0 | 3 verdicts; conservative default on failure |
| `_route_after_grader` | Conditional edge function | `pass`/`regenerate`/`web_search`; safe default |
| `build_query_graph` | LangGraph `StateGraph` + `MemorySaver` | 6 nodes; loop guard on `iteration_count` |

**Loop Guard:** `iteration_count >= settings.max_hallucination_retries` forces `action="web_search"`,
preventing infinite regeneration loops on hard queries.

**Two-Graph Architecture:**
- **Builder Graph** (Part 6) — offline/batch: PDF + DDL → Neo4j Knowledge Graph  
- **Query Graph** (Part 8) — online/latency-sensitive: user query → grounded answer